# 10年定着予測 - 113列でのハイパーパラメータ再探索とさらなる減量（42_）

**背景**: `40_` の `R6_lean`（**113列**）が Public **0.521729** で新最良になった
（`37_` D3 の441列 0.522659 から −0.00093）。
441列のうち328列（74%）を落としてもスコアが落ちなかったことで、2つの手が開いた。

| # | 手 | 根拠 |
|---|---|---|
| 1 | **113列に合わせたハイパーパラメータ再探索** | `A_PARAMS`（depth=4, l2=2.22）は**441列向けに探索した設定**。列数が1/4になれば最適な複雑さは変わるはず |
| 2 | **さらなる減量** | 113列のうち **80列は `agg`**。`40_` のLOO診断では `agg` を丸ごと落とすと検証が0.0107改善していた |

**重要**: 1は特徴量選択ではなく**モデル設定**の話なので、
`39_` で確定した「アブレーションの検証スコアはPublicを予測しない」には抵触しない。
2は減量の方向そのものが `40_` の Public で確認済みなので、次の端点を試す。

## 事前登録した仮説（外れたら正直に記録する）

**「113列では441列より最適 depth が上がる」** ——
特徴量が1/4になれば仮説空間が小さくなり、強い正則化の必要性が下がるため。
`38_` のマルチシードOptuna（441列）は depth=4, l2=2.18 と `A_PARAMS` とほぼ同じ領域に着地し、
改善幅 −0.002 はシードsd以下で**Publicに転移しなかった**。
**今回も depth=4 付近に戻るなら、再探索という手そのものが尽きていると判断する。**

## ベースライン

`40_` の `R6_lean` と完全に同一の113列。

| グループ | 列数 |
|---|---|
| `agg`（16指標 × 5統計 mean/std/early_mean/late_mean/slope） | 80 |
| `persona` | 21 |
| `derived` | 8 |
| `deptte` | 2 |
| `L2` | 2 |

## 実行構成

| config | 列数 | パラメータ | 反復数 | 位置づけ |
|---|---|---|---|---|
| `T0_ref_lean113` | 113 | `A_PARAMS` | 560 | **参照**。`R6_lean`(Public 0.521729)の再現 |
| `T1_retuned_iter560` | 113 | **再探索** | 560 | 本命1。T0からの変更が**パラメータだけ** |
| `T1b_retuned_iterfit` | 113 | **再探索** | 自前のES×1.25 | 本命1の手続き一定版 |
| `T2_lean113_itercurve` | 113 | `A_PARAMS` | 曲線の最適点 | **条件付き**（曲線の振れ幅がシードsdを超えたときだけ作る） |
| `S1_no_agg33` | **33** | `A_PARAMS` | 560 | 本命2。`agg` を全部落とす端点 |
| `S2_agg_mean49` | **49** | `A_PARAMS` | 560 | 中間点（`agg` は mean のみ） |

## 判定方法（事前登録）

- **採否は Public でのみ決める。** 検証（生存者535名）の分解能は±0.011で、
  ここで見込む効果（0.002〜0.005）は原理的に判定できない
- **`T1` 系は、再探索したパラメータが `A_PARAMS` と実質的に違い、かつマルチシードval改善が
  シードsdを超えた場合のみ提出する。** `38_` で「探索した検証セット上で有利に出るのは当然」と
  結論しているため、この条件を満たさないなら提出しない
- 足切り: `T0` から +0.02 以上悪化した構成は提出しない

## 実行環境

Google Colab Pro の **CPUハイメモリ**ランタイム。
113列なので441列より大幅に速い。反復数曲線21回 + Optuna 30試行×3シード + 最終学習約70回で
想定 **1〜1.5時間**。

> ⚠️ **ローカルMacで先行実行しないこと。** `27_` のチェックポイント同期事故を避けるため、
> Colabで直接実行する。やり直したい場合は `RESET_CHECKPOINT = True` にする。


In [58]:
!pip install -q catboost optuna

In [59]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [60]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [61]:
import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [62]:
SCRIPT_NAME = "42_lean113_retune_and_shrink"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# チェックポイント（日付非依存の固定パス。セッションをまたいで再開できるようにする）
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

RESET_CHECKPOINT = True  # Trueにすると既存チェックポイントを削除して最初から再計算する
if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("チェックポイントを削除しました（全構成を再計算します）")

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")

[2026-08-12 00:45:38] [INFO] === [42_lean113_retune_and_shrink] 実験開始 ===


INFO:42_lean113_retune_and_shrink:=== [42_lean113_retune_and_shrink] 実験開始 ===


[2026-08-12 00:45:38] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260812


INFO:42_lean113_retune_and_shrink:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260812


[2026-08-12 00:45:38] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/42_lean113_retune_and_shrink_checkpoint.csv


INFO:42_lean113_retune_and_shrink:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/42_lean113_retune_and_shrink_checkpoint.csv


[2026-08-12 00:45:38] [INFO] チェックポイントは未作成（新規実行）


INFO:42_lean113_retune_and_shrink:チェックポイントは未作成（新規実行）


In [63]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-12 00:45:39] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:42_lean113_retune_and_shrink:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-12 00:45:39] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:42_lean113_retune_and_shrink:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-12 00:45:39] [INFO] 定着率: 0.5647


INFO:42_lean113_retune_and_shrink:定着率: 0.5647


[2026-08-12 00:45:39] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:42_lean113_retune_and_shrink:Train IDs: 2761, Test IDs: 2502


## 0. 早期退職者の特定（改善3の前提）

`月末在籍状態 == "退職"` の行を持つ社員が、0-23ヶ月の観測期間中に退職した社員。
Train に129名（全員ラベル0）、**Test には0名**。

In [64]:
EARLY_LEAVER_IDS = set(train_monthly.loc[train_monthly["月末在籍状態"] == "退職", ID_COL].unique())
_test_early = set(test_monthly.loc[test_monthly["月末在籍状態"] == "退職", ID_COL].unique())

logger.info(f"Train 早期退職者: {len(EARLY_LEAVER_IDS)}名 / {len(train_ids)}名 ({len(EARLY_LEAVER_IDS)/len(train_ids):.1%})")
logger.info(f"Test  早期退職者: {len(_test_early)}名 / {len(test_ids)}名")
_y_idx = train_persona.set_index(ID_COL)[TARGET_COL]
logger.info(f"早期退職者のラベル平均: {_y_idx.loc[list(EARLY_LEAVER_IDS)].mean():.4f}（0.0のはず）")
logger.info(f"定着率: 全体 {y_train.mean():.4f} / 早期退職者を除く {_y_idx[~_y_idx.index.isin(EARLY_LEAVER_IDS)].mean():.4f}")
assert len(_test_early) == 0, "Testに早期退職者が存在する。EDA v6の前提が崩れているので調査すること"

[2026-08-12 00:45:40] [INFO] Train 早期退職者: 129名 / 2761名 (4.7%)


INFO:42_lean113_retune_and_shrink:Train 早期退職者: 129名 / 2761名 (4.7%)


[2026-08-12 00:45:40] [INFO] Test  早期退職者: 0名 / 2502名


INFO:42_lean113_retune_and_shrink:Test  早期退職者: 0名 / 2502名


[2026-08-12 00:45:40] [INFO] 早期退職者のラベル平均: 0.0000（0.0のはず）


INFO:42_lean113_retune_and_shrink:早期退職者のラベル平均: 0.0000（0.0のはず）


[2026-08-12 00:45:40] [INFO] 定着率: 全体 0.5647 / 早期退職者を除く 0.5923


INFO:42_lean113_retune_and_shrink:定着率: 全体 0.5647 / 早期退職者を除く 0.5923


## 1. 基本特徴量関数の定義（split非依存、`18_`〜`26_`と同一ロジック）

In [65]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")

✅ split非依存の基本特徴量関数定義完了


In [66]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")

[2026-08-12 00:45:40] [INFO] ------------------------------------------------------------


INFO:42_lean113_retune_and_shrink:------------------------------------------------------------


[2026-08-12 00:45:40] [INFO] split非依存の基本特徴量を生成中...


INFO:42_lean113_retune_and_shrink:split非依存の基本特徴量を生成中...


[2026-08-12 00:45:40] [INFO] ------------------------------------------------------------


INFO:42_lean113_retune_and_shrink:------------------------------------------------------------


[2026-08-12 00:53:53] [INFO] split非依存の基本特徴量生成完了


INFO:42_lean113_retune_and_shrink:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`18_`と同一・継続採用）

In [67]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")

[2026-08-12 00:53:53] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:42_lean113_retune_and_shrink:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-12 00:53:55] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:42_lean113_retune_and_shrink:入社時メモ: SVD累積寄与率=0.760


[2026-08-12 00:54:00] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:42_lean113_retune_and_shrink:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-12 00:54:03] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.421


INFO:42_lean113_retune_and_shrink:同僚からのフィードバック: SVD累積寄与率=0.421


[2026-08-12 00:54:03] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:42_lean113_retune_and_shrink:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`18_`の勝者を継続採用）

`18_`のステップAで、D_expanded（16指標）がD_original（6指標）・Dなしより2 split平均で最良と判明したため、
以降は常にD_expandedを使う（今回はDブロックの再比較は行わない）。

In [68]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")

[2026-08-12 00:54:03] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:42_lean113_retune_and_shrink:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-12 00:57:10] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:42_lean113_retune_and_shrink:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（split非依存、`18_`と同一）

In [69]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")

[2026-08-12 00:57:11] [INFO] Persona単位の基本特徴量を生成中...


INFO:42_lean113_retune_and_shrink:Persona単位の基本特徴量を生成中...


[2026-08-12 00:57:11] [INFO] Persona単位の基本特徴量処理完了


INFO:42_lean113_retune_and_shrink:Persona単位の基本特徴量処理完了


## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、v1=`27_`のPublic確認済み版 / v2=抽出拡張版）

`転居許容`フラグの抽出ロジックは`27_`と同一。`希望勤務地`の抽出のみ2種類を用意する：

- **v1**: `27_`・`25_`・`data_exploration_v3/v4/v5`と同一の正規表現（Public 0.529672で確認済み）
- **v2**: v1に加え、「◯◯を希望。」「◯◯勤務を希望。」「◯◯での勤務を希望。」パターンを追加で
  拾う拡張版。未抽出だった152件（train）を目視確認して発見した言い回し。カバー率が
  88.6%→94.1%（train）/ 95.0%（test）に向上し、ダブル悪条件の該当件数も342→354件に増加、
  効果量はp=1.4×10⁻³⁹→1.7×10⁻⁴³・オッズ比0.189→0.178とむしろ強まった。

In [70]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    return m.group(1).strip() if m else None


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    # reloc_ok_rawはobject dtype(True/False/None混在)のため、~演算子は使わず
    # 明示的な等価比較でTrue/False/欠損を扱う（欠損に対する~はTypeErrorになる）
    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    # 4値カテゴリ（決定木が交互作用を直接学習しやすいよう明示的にエンコード）
    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    # ダブル悪条件フラグ（EDA v5で確認した最も強いシグナル: 転居許容せず AND 勤務地不一致）
    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")
print("L_v1 ダブル悪条件:")
print(train_reloc_v1["転居x勤務地_ダブル悪条件_v1"].value_counts())
print("\nL_v2 ダブル悪条件:")
print(train_reloc_v2["転居x勤務地_ダブル悪条件_v2"].value_counts())

[2026-08-12 00:57:11] [INFO] 転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


INFO:42_lean113_retune_and_shrink:転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


[2026-08-12 00:57:11] [INFO] L_v1: Train (2761, 3), Test (2502, 3)


INFO:42_lean113_retune_and_shrink:L_v1: Train (2761, 3), Test (2502, 3)


[2026-08-12 00:57:11] [INFO] L_v2: Train (2761, 3), Test (2502, 3)


INFO:42_lean113_retune_and_shrink:L_v2: Train (2761, 3), Test (2502, 3)


L_v1 ダブル悪条件:
転居x勤務地_ダブル悪条件_v1
0    2419
1     342
Name: count, dtype: int64

L_v2 ダブル悪条件:
転居x勤務地_ダブル悪条件_v2
0    2407
1     354
Name: count, dtype: int64


## 6. 部署Target Encoding（リーク対策済）と `prepare_split` 関数

`extra_blocks`パラメータで`{"L1"}`/`{"L2"}`を指定し、ベースライン
（D_expanded + TF-IDF A_v1、`18_`の構成、Eなし）に対してL_v1・L_v2のいずれかを単体で追加できるようにする。

In [71]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None, exclude_early_from_val=True):
    '''指定した分割比率で特徴量を組み立てる。

    28_ からの変更点は2つだけ:
      - split_ratio=1.0 を許容（Train全件学習用。ag_tuningは空になる）
      - exclude_early_from_val=True のとき、検証セットから早期退職者を除く（改善3）
    特徴量の作り方そのものは 28_ と完全に同一。
    '''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L1" in extra_blocks:
        tf = tf.merge(train_reloc_v1, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v1, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    # --- 改善3: 検証セットから早期退職者を除く（学習側からは除かない） ---
    if exclude_early_from_val and len(ag_tuning) > 0:
        n_before = len(ag_tuning)
        ag_tuning = ag_tuning[~ag_tuning.index.isin(EARLY_LEAVER_IDS)]
        logger.info(f"  検証セット: {n_before} → {len(ag_tuning)}件（早期退職者{n_before - len(ag_tuning)}名を除外）")

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了（37_版: 全件学習・検証セット補正に対応）")

✅ 部署Target Encoding・prepare_split関数定義完了（37_版: 全件学習・検証セット補正に対応）


## 7. チェックポイント機能（`18_`〜`28_`をベースに、37_で固定スキーマ化）

`28_`までは全configが同じキーを持っていたが、37_ は A / BC / D で記録すべき情報が異なる。
キー構成がバラバラのまま `mode="a"` でCSVに追記すると列がずれて壊れるため、
`RESULT_SCHEMA` に揃えてから書き出す。

In [72]:
RESULT_SCHEMA = ["config", "n_features", "val_score", "val_score_all", "val_score_single",
                   "val_single_mean", "val_single_sd", "best_iter", "n_iterations",
                   "params", "submission_path"]

def make_row(**kwargs):
    """全configで同じ列構成のdictを作る。

    28_ は全configが同じキーを持っていたが、37_ は A / BC / D で必要な情報が異なる。
    キー構成がバラバラのままだと、mode="a" でCSVに追記した際に列がずれて壊れるため、
    固定スキーマに揃えてから書き出す。
    """
    unknown = set(kwargs) - set(RESULT_SCHEMA)
    assert not unknown, f"RESULT_SCHEMAに無いキー: {unknown}"
    row = {k: np.nan for k in RESULT_SCHEMA}
    row.update(kwargs)
    return row


def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=RESULT_SCHEMA)

def save_checkpoint_row(result):
    df = pd.DataFrame([result])[RESULT_SCHEMA]
    write_header = not CHECKPOINT_PATH.exists()
    df.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)

def run_or_resume(config_label, run_fn):
    checkpoint = load_checkpoint()
    existing = checkpoint[checkpoint["config"] == config_label]
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元: val_score={row['val_score']}")
        return row
    result = run_fn()
    save_checkpoint_row(result)
    return result

print("✅ チェックポイント関数定義完了（37_版: 固定スキーマで列ずれを防止）")

✅ チェックポイント関数定義完了（37_版: 固定スキーマで列ずれを防止）


## 8. モデル関数（37_版）

`28_`の `run_model_config` を3つに分解する。

- `tune_hyperparams`: Optunaで探索（探索空間は`18_`〜`28_`と完全に同一、n_trials=25）
- `fit_holdout`: 80/20で学習し、early stoppingで最良反復数を決める。シードを変えて複数回実行できる
- `fit_full_train`: **Train全件**で学習する（検証セットが無いので反復数は固定、early stoppingなし）

シード平均は、同一パラメータ・同一特徴量のままシードだけ変えたモデルの**予測確率を単純平均**する。
重みを一切学習しないので、`11_`/`12_`/`32_`で失敗した「OOFから重みを学習するアンサンブル」とは
別物であり、過去の教訓には抵触しない。

In [73]:
SEEDS = [42, 2024, 7, 1234, 99]          # 改善2: シード平均に使う5シード
N_TRIALS = 25                            # 18_〜28_と同一
ITER_SCALE_CANDIDATES = {"x125": 1.25, "x100": 1.00}   # 全件学習時の反復数スケール（2761/2208≒1.25）


def _feature_cols(df):
    return [c for c in df.columns if c not in ["入社日", TARGET_COL]]


def _xy(df, feature_cols):
    return df[feature_cols].fillna(-999), df[TARGET_COL]


def tune_hyperparams(ag_train, ag_val, n_trials=N_TRIALS):
    """Optunaでハイパーパラメータを探索（探索空間は18_〜28_と完全に同一）"""
    feature_cols = _feature_cols(ag_train)
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_train, feature_cols)
    X_va, y_va = _xy(ag_val, feature_cols)

    def objective(trial):
        params = {
            "depth": trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "iterations": 1000, "random_seed": SEED, "verbose": False,
            "cat_features": obj_cols, "early_stopping_rounds": 50, "task_type": "CPU",
        }
        model = cb.CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        return log_loss(y_va, model.predict_proba(X_va)[:, 1])

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    logger.info(f"  Optuna完了: best_value={study.best_value:.6f}, best_params={study.best_params}")
    return study.best_params


def fit_holdout(ag_train, ag_val, test_features, best_params, seeds):
    """80/20で学習。early stoppingで最良反復数を決め、シードごとの予測を返す"""
    feature_cols = _feature_cols(ag_train)
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_train, feature_cols)
    X_va, y_va = _xy(ag_val, feature_cols)
    X_test = test_features[feature_cols].fillna(-999)

    val_preds, test_preds, best_iters = [], [], []
    for seed in seeds:
        model = cb.CatBoostClassifier(
            **best_params, iterations=3000, random_seed=seed, verbose=False,
            cat_features=obj_cols, early_stopping_rounds=100, task_type="CPU",
        )
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        vp = model.predict_proba(X_va)[:, 1]
        val_preds.append(vp)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        best_iters.append(model.get_best_iteration())
        logger.info(f"  seed={seed}: val_logloss={log_loss(y_va, vp):.6f}, best_iteration={best_iters[-1]}")

    return {
        "val_preds": np.array(val_preds), "test_preds": np.array(test_preds),
        "best_iters": best_iters, "y_val": y_va.values, "feature_cols": feature_cols,
    }


def fit_full_train(ag_full, test_features, best_params, n_iterations, seeds):
    """Train全件で学習（改善1）。検証セットが無いので反復数は固定、early stoppingなし"""
    feature_cols = _feature_cols(ag_full)
    obj_cols = [c for c in feature_cols if ag_full[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_full, feature_cols)
    X_test = test_features[feature_cols].fillna(-999)

    test_preds = []
    for seed in seeds:
        model = cb.CatBoostClassifier(
            **best_params, iterations=int(n_iterations), random_seed=seed, verbose=False,
            cat_features=obj_cols, task_type="CPU",
        )
        model.fit(X_tr, y_tr)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        logger.info(f"  seed={seed}: 全件学習完了（iterations={int(n_iterations)}）")
    return np.array(test_preds)


def save_submission(test_index, preds, config_label):
    path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    pd.DataFrame({ID_COL: test_index, TARGET_COL: preds}).to_csv(path, index=False, header=False)
    logger.info(f"  提出ファイル: {path.name}（予測平均={preds.mean():.4f}）")
    return str(path)


print("✅ モデル関数定義完了（tune_hyperparams / fit_holdout / fit_full_train）")

✅ モデル関数定義完了（tune_hyperparams / fit_holdout / fit_full_train）


## 9. 特徴量の組み立て

ブロックは `L2`（= `28_`の `L_v2_extended`、現在の最良）に固定する。

- `split_80_20` × 検証=全体 → config A（`28_`の完全再現）
- `split_80_20` × 検証=生存者のみ → config B / C
- `split_100`（全件） → config D / D2

In [74]:
BLOCK = {"L2"}   # 28_のL_v2_extended（現在の最良）に固定

logger.info("=" * 60)
logger.info("[A用] split_80_20 / 検証=全体（28_と同一）")
ag_train_80, ag_val_all, test_features = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=False)

logger.info("[B,C用] split_80_20 / 検証=生存者のみ")
ag_train_80b, ag_val_surv, _ = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("[D用] 全件学習（検証セットなし）")
ag_full, ag_empty, test_features_full = prepare_split(1.0, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("-" * 60)
logger.info(f"A: train={len(ag_train_80)}, val={len(ag_val_all)}（早期退職者を含む）")
logger.info(f"B/C: train={len(ag_train_80b)}, val={len(ag_val_surv)}（生存者のみ）")
logger.info(f"D: train={len(ag_full)}（全件）, val={len(ag_empty)}（空）")
logger.info(f"特徴量数: {len(_feature_cols(ag_train_80))}")

# 生存者マスク（Aの検証予測を生存者だけで採点し直すのに使う）
SURV_MASK_A = ~ag_val_all.index.isin(EARLY_LEAVER_IDS)
assert len(ag_train_80) == len(ag_train_80b), "A と B/C の学習データは同一のはず"
assert len(ag_empty) == 0, "全件学習のときは検証セットが空のはず"

[2026-08-12 00:57:12] [INFO] ============================================================


INFO:42_lean113_retune_and_shrink:============================================================


[2026-08-12 00:57:12] [INFO] [A用] split_80_20 / 検証=全体（28_と同一）


INFO:42_lean113_retune_and_shrink:[A用] split_80_20 / 検証=全体（28_と同一）


[2026-08-12 00:57:12] [INFO] [B,C用] split_80_20 / 検証=生存者のみ


...

INFO:42_lean113_retune_and_shrink:B/C: train=2208, val=535（生存者のみ）


[2026-08-12 00:57:13] [INFO] D: train=2761（全件）, val=0（空）


INFO:42_lean113_retune_and_shrink:D: train=2761（全件）, val=0（空）


[2026-08-12 00:57:13] [INFO] 特徴量数: 441


INFO:42_lean113_retune_and_shrink:特徴量数: 441


## 10. 特徴量グループの棚卸し（`40_` から移植）

In [75]:
# ※ このセルは 40_feature_reduction.ipynb から移植したもの（内容は同一）。

# ============================================================
# 特徴量グループの棚卸し
#   prepare_split() が merge している元フレームごとに列を分類する。
#   「どのグループにも属さない列」「2グループに重複する列」が出たら
#   減量の定義がずれているのでassertで止める。
# ============================================================

ALL_FEATS = set(_feature_cols(ag_train_80))

# prepare_split() 内で生成される派生列（元フレームを持たないのでここに明示）
DERIVED_COLS = [
    "残業時間_mean_job_deviation", "研修時間_mean_job_deviation", "360度評価_親和度_mean_job_deviation",
    "研修時間_職種比", "研修時間_区分比",
    "初任給_等級内偏差", "初任給_区分内偏差", "月例給与_等級内偏差",
]
# 部署Target Encoding が生む列（create_department_target_encoding の出力）
DEPT_TE_COLS = ["dept_target_enc", "dept_size"]


def _cols_of(df):
    return [c for c in df.columns if c != ID_COL]


_RAW_GROUPS = {
    "persona":   [c for c in train_persona.columns if c not in (ID_COL, TARGET_COL)],
    "agg":       _cols_of(train_monthly_agg),
    "catchange": _cols_of(train_cat_change),
    "missing":   _cols_of(train_missing),
    "domain":    _cols_of(train_domain),
    "advstats":  _cols_of(train_advanced_stats),
    "cluster":   _cols_of(train_cluster),
    "deptte":    DEPT_TE_COLS,
    "edafeat":   _cols_of(train_eda_feats),
    "mgr":       _cols_of(train_mgr),
    "quarterly": _cols_of(train_quarterly_exp),
    "tfidf":     [c for _df in tfidf_train_list for c in _cols_of(_df)],
    "L2":        _cols_of(train_reloc_v2),
    "derived":   DERIVED_COLS,
}

# 実際に特徴量として残っている列だけに絞る（drop_colsで消えたものを自動的に除外）
FEATURE_GROUPS = {g: [c for c in cols if c in ALL_FEATS] for g, cols in _RAW_GROUPS.items()}
ALL_GROUPS = set(FEATURE_GROUPS)

_covered = [c for cols in FEATURE_GROUPS.values() for c in cols]
_dupes = sorted({c for c in _covered if _covered.count(c) > 1})
assert not _dupes, f"複数グループに重複している列: {_dupes}"
_orphans = sorted(ALL_FEATS - set(_covered))
assert not _orphans, f"どのグループにも属さない列: {_orphans}"

print(f"特徴量 合計 {len(ALL_FEATS)} 列")
print("-" * 52)
for g in sorted(FEATURE_GROUPS, key=lambda x: -len(FEATURE_GROUPS[x])):
    print(f"  {g:<10s} {len(FEATURE_GROUPS[g]):>4d} 列   例: {FEATURE_GROUPS[g][:2]}")
print("-" * 52)
print("✅ グループ分類は全列を過不足なく覆っている")


特徴量 合計 441 列
----------------------------------------------------
  agg         224 列   例: ['残業時間_mean', '残業時間_std']
  quarterly    80 列   例: ['残業時間_q1_mean_exp', '残業時間_q2_mean_exp']
  tfidf        45 列   例: ['入社時メモ_tfidf_svd_0', '入社時メモ_tfidf_svd_1']
  advstats     25 列   例: ['残業時間_skew', '残業時間_kurtosis']
  persona      21 列   例: ['入社区分', '入社時年齢']
  catchange    14 列   例: ['部署ID_changes', '部署ID_unique_count']
  edafeat      11 列   例: ['欠勤発生月数', '欠勤_最長連続月数']
  derived       8 列   例: ['残業時間_mean_job_deviation', '研修時間_mean_job_deviation']
  missing       4 列   例: ['360度評価_親和度_missing_rate', '360度評価_信頼度_missing_rate']
  domain        3 列   例: ['engagement_score', 'overtime_stability']
  deptte        2 列   例: ['dept_target_enc', 'dept_size']
  L2            2 列   例: ['転居x勤務地_状態_v2', '転居x勤務地_ダブル悪条件_v2']
  cluster       1 列   例: ['cluster']
  mgr           1 列   例: ['初期上司_部下数']
----------------------------------------------------
✅ グループ分類は全列を過不足なく覆っている


In [76]:
# ※ このセルは 40_feature_reduction.ipynb から移植したもの（内容は同一）。

# ============================================================
# 月次集約(agg)の「指標 × 統計」分解
#   create_monthly_aggregation_features が作る 16指標 × 14統計 を分解し、
#   冗長な統計を落とせるようにする。
# ============================================================

AGG_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]
AGG_ALL_STATS = [
    "mean", "std", "min", "max", "median", "cv",
    "early_mean", "mid_mean", "late_mean", "late_minus_early", "late_early_ratio",
    "slope", "diff", "ratio",
]

# 残す統計。冗長性を根拠に選ぶ（検証スコアで選んでいない）:
#   median←mean と重複 / cv←std/mean の比 / min,max←外れ値1点 /
#   mid_mean←early,lateから内挿可能 / late_minus_early,late_early_ratio,diff,ratio←slopeと同義
AGG_KEEP_STATS = {"mean", "std", "early_mean", "late_mean", "slope"}


def _agg_stat(col):
    """agg列名を (指標, 統計) に分解して統計名を返す。最長一致で指標を特定する。"""
    best = None
    for m in AGG_METRICS:
        if col.startswith(m + "_") and (best is None or len(m) > len(best)):
            best = m
    if best is None:
        return None
    return col[len(best) + 1:]


_unmapped = [c for c in FEATURE_GROUPS["agg"] if _agg_stat(c) not in AGG_ALL_STATS]
assert not _unmapped, f"指標×統計に分解できないagg列: {_unmapped}"

AGG_SLIM_COLS = [c for c in FEATURE_GROUPS["agg"] if _agg_stat(c) in AGG_KEEP_STATS]
print(f"agg: {len(FEATURE_GROUPS['agg'])} 列 → 統計を{sorted(AGG_KEEP_STATS)}に限定すると {len(AGG_SLIM_COLS)} 列")
print(f"落とす統計: {sorted(set(AGG_ALL_STATS) - AGG_KEEP_STATS)}")


agg: 224 列 → 統計を['early_mean', 'late_mean', 'mean', 'slope', 'std']に限定すると 80 列
落とす統計: ['cv', 'diff', 'late_early_ratio', 'late_minus_early', 'max', 'median', 'mid_mean', 'min', 'ratio']


## 11. ベースラインと特徴量セットの定義

In [77]:
# ============================================================
# ベースライン: 40_ の R6_lean と同一の113列
# ============================================================

CORE_GROUPS = {"persona", "agg", "deptte", "derived", "L2"}

FEATURE_SPECS = {
    # 40_ R6_lean（Public 0.521729）と同一
    "lean113":     {"groups": CORE_GROUPS, "agg_stats": AGG_KEEP_STATS},
    # agg を丸ごと落とす端点
    "no_agg33":    {"groups": CORE_GROUPS - {"agg"}, "agg_stats": None},
    # agg は mean のみ残す中間点
    "agg_mean49":  {"groups": CORE_GROUPS, "agg_stats": {"mean"}},
}

BASE_SPEC_NAME = "lean113"


def cols_for(spec_name, df):
    """特徴量セット名から列リストを作る（元データフレームの列順を保つ）"""
    spec = FEATURE_SPECS[spec_name]
    keep = set()
    for g in spec["groups"]:
        if g == "agg" and spec["agg_stats"] is not None:
            keep |= {c for c in FEATURE_GROUPS["agg"] if _agg_stat(c) in spec["agg_stats"]}
        else:
            keep |= set(FEATURE_GROUPS[g])
    return [c for c in _feature_cols(df) if c in keep]


print(f"{'特徴量セット':<14s} {'列数':>5s}  内訳")
print("-" * 72)
for _n in FEATURE_SPECS:
    _c = cols_for(_n, ag_full)
    _bd = {g: len([x for x in _c if x in set(FEATURE_GROUPS[g])]) for g in sorted(ALL_GROUPS)}
    _bd = {g: v for g, v in _bd.items() if v}
    print(f"{_n:<14s} {len(_c):>5d}  {_bd}")

# 40_ の実行結果と一致するかの自己照合
_expected = {"lean113": 113, "no_agg33": 33, "agg_mean49": 49}
for _n, _e in _expected.items():
    _got = len(cols_for(_n, ag_full))
    assert _got == _e, f"{_n}: {_got}列（{_e}列のはず）。グループ定義を確認すること"
    assert cols_for(_n, ag_train_80b) == cols_for(_n, ag_full), \
        f"{_n}: 80%学習と全件学習で列が食い違っている"
print()
print("✅ 113 / 33 / 49 列を確認（113は 40_ R6_lean と同一）")


特徴量セット            列数  内訳
------------------------------------------------------------------------
lean113          113  {'L2': 2, 'agg': 80, 'deptte': 2, 'derived': 8, 'persona': 21}
no_agg33          33  {'L2': 2, 'deptte': 2, 'derived': 8, 'persona': 21}
agg_mean49        49  {'L2': 2, 'agg': 16, 'deptte': 2, 'derived': 8, 'persona': 21}

✅ 113 / 33 / 49 列を確認（113は 40_ R6_lean と同一）


## 12. 設定の事前登録

In [78]:
# ============================================================
# 設定の事前登録
# ============================================================

A_PARAMS = {
    "depth": 4,
    "learning_rate": 0.03518359458951149,
    "l2_leaf_reg": 2.217690447016724,
    "border_count": 218,
    "bagging_temperature": 0.6787467566574921,
    "random_strength": 1.438494697238285,
}

ITER_BASE   = 560   # 40_ R6_lean と同一（441列の80/20でのbest_iter 449 × 1.25 に由来）
ITER_SCALE  = 1.25  # 80%学習(2208件) → 全件(2761件) のスケール

SEEDS_SUB   = [42, 2024, 7, 1234, 99]                      # 提出用。D3・40_と同一
SEEDS_VAL   = [42, 2024, 7, 1234, 99, 555, 31337, 2718]    # 検証用8シード（40_と同一）
TUNE_SEEDS  = [42, 2024, 7]                                # Optunaの目的関数（38_と同一）
N_TRIALS    = 30

ITER_GRID = [250, 350, 450, 560, 700, 900, 1200]

VAL_REJECT_MARGIN = 0.02   # T0からこれ以上悪化した構成は提出しない

# T1系を提出する条件（38_の教訓: 探索した検証セット上で有利に出るのは当然）
#   (1) 再探索パラメータが A_PARAMS と実質的に異なる
#   (2) マルチシードvalの改善幅がシードsdを超える
# 両方を満たさなければ提出しない。
TUNE_SUBMIT_REQUIRES_BOTH = True

print("A_PARAMS（441列向けに探索された現行設定）:")
for k, v in A_PARAMS.items():
    print(f"  {k:<22s} {v}")
print(f"\n事前登録した仮説: 113列では depth が 4 より大きくなるはず")


A_PARAMS（441列向けに探索された現行設定）:
  depth                  4
  learning_rate          0.03518359458951149
  l2_leaf_reg            2.217690447016724
  border_count           218
  bagging_temperature    0.6787467566574921
  random_strength        1.438494697238285

事前登録した仮説: 113列では depth が 4 より大きくなるはず


## 13. モデル関数

In [79]:
# ============================================================
# モデル関数
# ============================================================

def _obj_cols(df, feature_cols):
    return [c for c in feature_cols if df[c].dtype == "object"]


def fit_fixed(ag_train, ag_val, feature_cols, params, n_iter, seeds):
    """反復数固定で学習し、シードごとの検証予測を返す"""
    oc = _obj_cols(ag_train, feature_cols)
    X_tr, y_tr = ag_train[feature_cols].fillna(-999), ag_train[TARGET_COL]
    X_va, y_va = ag_val[feature_cols].fillna(-999), ag_val[TARGET_COL]
    vps = []
    for seed in seeds:
        m = cb.CatBoostClassifier(**params, iterations=int(n_iter), random_seed=seed,
                                  verbose=False, cat_features=oc, task_type="CPU")
        m.fit(X_tr, y_tr)
        vps.append(m.predict_proba(X_va)[:, 1])
    vps = np.array(vps)
    singles = [log_loss(y_va, v) for v in vps]
    return {"val_seedavg": float(log_loss(y_va, vps.mean(axis=0))),
            "val_single_mean": float(np.mean(singles)),
            "val_single_sd": float(np.std(singles)),
            "val_preds": vps, "y_val": y_va.values}


def best_iter_es(ag_train, ag_val, feature_cols, params, seeds):
    """early stoppingでの最適反復数（平均）"""
    oc = _obj_cols(ag_train, feature_cols)
    X_tr, y_tr = ag_train[feature_cols].fillna(-999), ag_train[TARGET_COL]
    X_va, y_va = ag_val[feature_cols].fillna(-999), ag_val[TARGET_COL]
    its = []
    for seed in seeds:
        m = cb.CatBoostClassifier(**params, iterations=3000, random_seed=seed, verbose=False,
                                  cat_features=oc, early_stopping_rounds=100, task_type="CPU")
        m.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        its.append(m.get_best_iteration())
    return float(np.mean(its))


def fit_full(ag_full_, test_feats, feature_cols, params, n_iter, seeds):
    """Train全件で学習して Test を予測"""
    oc = _obj_cols(ag_full_, feature_cols)
    X_tr, y_tr = ag_full_[feature_cols].fillna(-999), ag_full_[TARGET_COL]
    X_te = test_feats[feature_cols].fillna(-999)
    tps = []
    for seed in seeds:
        m = cb.CatBoostClassifier(**params, iterations=int(n_iter), random_seed=seed,
                                  verbose=False, cat_features=oc, task_type="CPU")
        m.fit(X_tr, y_tr)
        tps.append(m.predict_proba(X_te)[:, 1])
        logger.info(f"    seed={seed}: 全件学習完了")
    return np.array(tps)


def tune_multiseed(ag_train, ag_val, feature_cols, n_trials=N_TRIALS, tune_seeds=TUNE_SEEDS):
    """マルチシード目的関数でのOptuna探索（38_と同一方式）。

    38_ で、単一シード目的関数は検証セットを3.3%変えただけで別領域へ飛ぶことが
    分かっている。各trialで複数シード学習して平均を取ることで探索が安定する。
    探索空間は 18_〜38_ と完全に同一。
    """
    oc = _obj_cols(ag_train, feature_cols)
    X_tr, y_tr = ag_train[feature_cols].fillna(-999), ag_train[TARGET_COL]
    X_va, y_va = ag_val[feature_cols].fillna(-999), ag_val[TARGET_COL]

    def objective(trial):
        params = {
            "depth": trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        }
        scores = []
        for seed in tune_seeds:
            m = cb.CatBoostClassifier(**params, iterations=1000, random_seed=seed, verbose=False,
                                      cat_features=oc, early_stopping_rounds=50, task_type="CPU")
            m.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
            scores.append(log_loss(y_va, m.predict_proba(X_va)[:, 1]))
        return float(np.mean(scores))

    study = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    logger.info(f"  Optuna完了: best_value={study.best_value:.6f}")
    logger.info(f"  best_params={study.best_params}")
    return study.best_params, float(study.best_value)


print("✅ モデル関数定義完了")


✅ モデル関数定義完了


In [80]:
# ============================================================
# チェックポイント（42_用にスキーマを差し替える）
# ============================================================

RESULT_SCHEMA = [
    "config", "feature_set", "n_features", "params_name", "params",
    "val_seedavg", "val_single_mean", "val_single_sd", "best_iter_es",
    "n_iterations", "n_train", "pred_mean", "submission_path",
]


def run_or_resume(config_label, run_fn):
    ck = load_checkpoint()
    existing = ck[ck["config"] == config_label] if len(ck) else ck
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元: val_seedavg={row.get('val_seedavg')}")
        return row
    r = run_fn()
    save_checkpoint_row(r)
    return r


_probe = make_row(config="__probe__", n_features=1)
assert list(_probe.keys()) == RESULT_SCHEMA, "make_rowが新スキーマを見ていない"
_rejected = False
try:
    make_row(config="x", val_score=0.5)   # 旧(37_)スキーマのキー
except AssertionError as _e:
    _rejected = "RESULT_SCHEMA" in str(_e)
assert _rejected, "旧スキーマのキーが素通りした"
print("✅ チェックポイントを42_スキーマに差し替え完了")


✅ チェックポイントを42_スキーマに差し替え完了


## A. 113列での反復数曲線（診断）

In [81]:
# ============================================================
# 第A節: 113列での反復数曲線（診断）
#   560 は441列の80/20で決めた値。列数が1/4になっても妥当か確認する。
#   事前登録: 曲線の振れ幅が典型シードsdを超えたときだけ T2 を作る。
# ============================================================

CURVE_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_iter_curve.csv"
_feats_base = cols_for(BASE_SPEC_NAME, ag_train_80b)

if CURVE_PATH.exists():
    curve = pd.read_csv(CURVE_PATH)
    logger.info(f"反復数曲線をチェックポイントから復元（{len(curve)}点）")
else:
    rows = []
    for n_it in ITER_GRID:
        r = fit_fixed(ag_train_80b, ag_val_surv, _feats_base, A_PARAMS, n_it, TUNE_SEEDS)
        rows.append({"iterations": n_it, "val_seedavg": r["val_seedavg"],
                     "val_single_mean": r["val_single_mean"], "val_single_sd": r["val_single_sd"]})
        logger.info(f"  iterations={n_it:>5d}: シード平均 {r['val_seedavg']:.6f} "
                    f"（単一 {r['val_single_mean']:.6f} ± {r['val_single_sd']:.6f}）")
    curve = pd.DataFrame(rows)
    curve.to_csv(CURVE_PATH, index=False)

print(curve.round(6).to_string(index=False))

TYPICAL_SD = float(curve["val_single_sd"].median())
_best_row = curve.loc[curve["val_seedavg"].idxmin()]
ITER_BEST_80 = int(_best_row["iterations"])
_span = float(curve["val_seedavg"].max() - curve["val_seedavg"].min())

print()
print(f"  最小点            : iterations={ITER_BEST_80}（val {_best_row['val_seedavg']:.6f}）")
print(f"  曲線全体の振れ幅  : {_span:.6f}")
print(f"  典型シードsd      : {TYPICAL_SD:.6f}")
print(f"  現行の560でのval  : {float(curve.loc[curve['iterations'] == 560, 'val_seedavg'].iloc[0]):.6f}")

# 最小点付近だけの振れ幅（両端の極端な値に引きずられないように）
_mid = curve[(curve["iterations"] >= 350) & (curve["iterations"] <= 900)]
_span_mid = float(_mid["val_seedavg"].max() - _mid["val_seedavg"].min())
print(f"  350〜900の振れ幅  : {_span_mid:.6f}")

MAKE_T2 = (_span_mid > TYPICAL_SD) and (ITER_BEST_80 != 560)
if MAKE_T2:
    print(f"\n  → 振れ幅がシードsdを超え、最小点も560でない。T2（iterations={int(ITER_BEST_80 * ITER_SCALE)}）を作る")
else:
    print(f"\n  → 350〜900は平坦（振れ幅 {_span_mid:.6f} ≤ シードsd {TYPICAL_SD:.6f}）"
          f"または最小点が560。38_と同じ結論なのでT2は作らない")


[2026-08-12 00:57:19] [INFO]   iterations=  250: シード平均 0.519350 （単一 0.520358 ± 0.002849）


INFO:42_lean113_retune_and_shrink:  iterations=  250: シード平均 0.519350 （単一 0.520358 ± 0.002849）


[2026-08-12 00:57:25] [INFO]   iterations=  350: シード平均 0.519086 （単一 0.520446 ± 0.001721）


INFO:42_lean113_retune_and_shrink:  iterations=  350: シード平均 0.519086 （単一 0.520446 ± 0.001721）


[2026-08-12 00:57:33] [INFO]   iterations=  450: シード平均 0.515319 （単一 0.517009 ± 0.001105）


...

[2026-08-12 00:58:11] [INFO]   iterations=  900: シード平均 0.522313 （単一 0.525750 ± 0.003725）


INFO:42_lean113_retune_and_shrink:  iterations=  900: シード平均 0.522313 （単一 0.525750 ± 0.003725）


[2026-08-12 00:58:34] [INFO]   iterations= 1200: シード平均 0.527221 （単一 0.531491 ± 0.005009）


INFO:42_lean113_retune_and_shrink:  iterations= 1200: シード平均 0.527221 （単一 0.531491 ± 0.005009）


 iterations  val_seedavg  val_single_mean  val_single_sd
        250     0.519350         0.520358       0.002849
        350     0.519086         0.520446       0.001721
        450     0.515319         0.517009       0.001105
        560     0.514226         0.516394       0.001461
        700     0.515619         0.518348       0.002058
        900     0.522313         0.525750       0.003725
       1200     0.527221         0.531491       0.005009

  最小点            : iterations=560（val 0.514226）
  曲線全体の振れ幅  : 0.012996
  典型シードsd      : 0.002058
  現行の560でのval  : 0.514226
  350〜900の振れ幅  : 0.008087

  → 350〜900は平坦（振れ幅 0.008087 ≤ シードsd 0.002058）または最小点が560。38_と同じ結論なのでT2は作らない


## B. ハイパーパラメータ再探索

In [82]:
# ============================================================
# 第B節: 113列でのハイパーパラメータ再探索（マルチシード目的関数）
# ============================================================

TUNE_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_tuned_params.json"

if TUNE_PATH.exists():
    _t = json.loads(TUNE_PATH.read_text())
    NEW_PARAMS, NEW_VALUE = _t["params"], _t["value"]
    logger.info("再探索結果をチェックポイントから復元")
else:
    logger.info(f"[Optuna] {N_TRIALS}試行 × {len(TUNE_SEEDS)}シード / {len(_feats_base)}列")
    NEW_PARAMS, NEW_VALUE = tune_multiseed(ag_train_80b, ag_val_surv, _feats_base)
    TUNE_PATH.write_text(json.dumps({"params": NEW_PARAMS, "value": NEW_VALUE}, ensure_ascii=False))

# 同じ目的関数で A_PARAMS を評価して比較する（探索値と同じ土俵に乗せる）
REF_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_ref_value.json"
if REF_PATH.exists():
    REF_VALUE = json.loads(REF_PATH.read_text())["value"]
else:
    _oc = _obj_cols(ag_train_80b, _feats_base)
    _Xtr, _ytr = ag_train_80b[_feats_base].fillna(-999), ag_train_80b[TARGET_COL]
    _Xva, _yva = ag_val_surv[_feats_base].fillna(-999), ag_val_surv[TARGET_COL]
    _sc = []
    for _s in TUNE_SEEDS:
        _m = cb.CatBoostClassifier(**A_PARAMS, iterations=1000, random_seed=_s, verbose=False,
                                   cat_features=_oc, early_stopping_rounds=50, task_type="CPU")
        _m.fit(_Xtr, _ytr, eval_set=(_Xva, _yva), use_best_model=True)
        _sc.append(log_loss(_yva, _m.predict_proba(_Xva)[:, 1]))
    REF_VALUE = float(np.mean(_sc))
    REF_PATH.write_text(json.dumps({"value": REF_VALUE}))

print("=" * 72)
print("ハイパーパラメータ再探索の結果（113列）")
print("=" * 72)
print(f"{'パラメータ':<22s} {'A_PARAMS(441列向け)':>22s} {'再探索(113列)':>18s}")
print("-" * 66)
for k in A_PARAMS:
    print(f"{k:<22s} {A_PARAMS[k]:>22.6f} {NEW_PARAMS[k]:>18.6f}")
print("-" * 66)
print(f"{'マルチシードval':<22s} {REF_VALUE:>22.6f} {NEW_VALUE:>18.6f}")
print(f"{'改善幅':<22s} {'':>22s} {NEW_VALUE - REF_VALUE:>+18.6f}")

print()
print("=" * 72)
print("事前登録した仮説の検証: 「113列では depth が 4 より大きくなるはず」")
print("=" * 72)
if NEW_PARAMS["depth"] > A_PARAMS["depth"]:
    print(f"  ✅ 的中: depth {A_PARAMS['depth']} → {NEW_PARAMS['depth']}")
    print("     列数が減って正則化の必要性が下がったという読みと整合する。")
elif NEW_PARAMS["depth"] == A_PARAMS["depth"]:
    print(f"  ❌ 外れ: depth は {NEW_PARAMS['depth']} のまま。")
    print("     38_（441列）と同じく A_PARAMS 付近に戻った。再探索という手は尽きている可能性が高い。")
else:
    print(f"  ❌ 外れ（逆方向）: depth {A_PARAMS['depth']} → {NEW_PARAMS['depth']}")

# --- 提出条件の判定（事前登録） ---
_gap = REF_VALUE - NEW_VALUE     # 正なら改善
_typ_sd = TYPICAL_SD
_params_differ = (NEW_PARAMS["depth"] != A_PARAMS["depth"]) or \
                 (abs(np.log(NEW_PARAMS["l2_leaf_reg"] / A_PARAMS["l2_leaf_reg"])) > np.log(2))
_gap_ok = _gap > _typ_sd

print()
print("=" * 72)
print("提出条件（事前登録: 両方を満たすときのみ T1 系を提出）")
print("=" * 72)
print(f"  (1) パラメータが実質的に異なる（depth変化 or L2が2倍以上変化）: "
      f"{'✅ 満たす' if _params_differ else '❌ 満たさない'}")
print(f"  (2) 改善幅 {_gap:+.6f} > 典型シードsd {_typ_sd:.6f}: "
      f"{'✅ 満たす' if _gap_ok else '❌ 満たさない'}")
SUBMIT_TUNED = bool(_params_differ and _gap_ok) if TUNE_SUBMIT_REQUIRES_BOTH else True
print(f"\n  → T1系の提出: {'する' if SUBMIT_TUNED else 'しない（38_と同じ結論）'}")
print("     ※ 学習と提出ファイルの生成は条件に関わらず行う（結果を記録に残すため）")


[2026-08-12 00:58:34] [INFO] [Optuna] 30試行 × 3シード / 113列


INFO:42_lean113_retune_and_shrink:[Optuna] 30試行 × 3シード / 113列


[2026-08-12 01:06:30] [INFO]   Optuna完了: best_value=0.515181


INFO:42_lean113_retune_and_shrink:  Optuna完了: best_value=0.515181


[2026-08-12 01:06:30] [INFO]   best_params={'depth': 5, 'learning_rate': 0.014021383758682313, 'l2_leaf_reg': 0.6603357701624626, 'border_count': 158, 'bagging_temperature': 0.5180728776840264, 'random_strength': 1.9201954024723764}


INFO:42_lean113_retune_and_shrink:  best_params={'depth': 5, 'learning_rate': 0.014021383758682313, 'l2_leaf_reg': 0.6603357701624626, 'border_count': 158, 'bagging_temperature': 0.5180728776840264, 'random_strength': 1.9201954024723764}


ハイパーパラメータ再探索の結果（113列）
パラメータ                        A_PARAMS(441列向け)          再探索(113列)
------------------------------------------------------------------
depth                                4.000000           5.000000
learning_rate                        0.035184           0.014021
l2_leaf_reg                          2.217690           0.660336
border_count                       218.000000         158.000000
bagging_temperature                  0.678747           0.518073
random_strength                      1.438495           1.920195
------------------------------------------------------------------
マルチシードval                            0.517234           0.515181
改善幅                                                    -0.002053

事前登録した仮説の検証: 「113列では depth が 4 より大きくなるはず」
  ✅ 的中: depth 4 → 5
     列数が減って正則化の必要性が下がったという読みと整合する。

提出条件（事前登録: 両方を満たすときのみ T1 系を提出）
  (1) パラメータが実質的に異なる（depth変化 or L2が2倍以上変化）: ✅ 満たす
  (2) 改善幅 +0.002053 > 典型シードsd 0.002058: ❌ 満たさない

  → T1系の提出: しない（38_と同じ結論）
     

## C. 実行する構成の確定

In [83]:
# ============================================================
# 第C節: 実行する構成の確定
# ============================================================

_iters_new = int(round(best_iter_es(ag_train_80b, ag_val_surv, _feats_base, NEW_PARAMS, TUNE_SEEDS)
                       * ITER_SCALE))
logger.info(f"再探索パラメータの ES best_iter × {ITER_SCALE} = {_iters_new}")

CONFIGS = {
    # 参照: 40_ R6_lean（Public 0.521729）の再現
    "T0_ref_lean113":      {"features": "lean113",    "params": A_PARAMS,   "iters": ITER_BASE,
                            "params_name": "A_PARAMS"},
    # 本命1: T0からの変更がパラメータだけ
    "T1_retuned_iter560":  {"features": "lean113",    "params": NEW_PARAMS, "iters": ITER_BASE,
                            "params_name": "NEW"},
    # 本命1の手続き一定版: 反復数も自前のESから決める（D3と同じ手続き）
    "T1b_retuned_iterfit": {"features": "lean113",    "params": NEW_PARAMS, "iters": _iters_new,
                            "params_name": "NEW"},
    # 本命2: aggを全部落とす端点
    "S1_no_agg33":         {"features": "no_agg33",   "params": A_PARAMS,   "iters": ITER_BASE,
                            "params_name": "A_PARAMS"},
    # 中間点
    "S2_agg_mean49":       {"features": "agg_mean49", "params": A_PARAMS,   "iters": ITER_BASE,
                            "params_name": "A_PARAMS"},
}

if MAKE_T2:
    CONFIGS["T2_lean113_itercurve"] = {
        "features": "lean113", "params": A_PARAMS,
        "iters": int(ITER_BEST_80 * ITER_SCALE), "params_name": "A_PARAMS"}

# T1b が T1 と同じ反復数なら冗長
if CONFIGS["T1b_retuned_iterfit"]["iters"] == ITER_BASE:
    del CONFIGS["T1b_retuned_iterfit"]
    print("再探索パラメータのES反復数が560と一致したため T1b は T1 と同一。スキップする。")

print(f"{'config':<22s} {'特徴量':<12s} {'列数':>5s} {'params':<10s} {'iters':>6s}")
print("-" * 62)
for _n, _s in CONFIGS.items():
    print(f"{_n:<22s} {_s['features']:<12s} {len(cols_for(_s['features'], ag_full)):>5d} "
          f"{_s['params_name']:<10s} {_s['iters']:>6d}")

assert CONFIGS["T0_ref_lean113"]["iters"] == 560, "T0は R6_lean と同じ560でなければ再現にならない"
assert len(cols_for("lean113", ag_full)) == 113


[2026-08-12 01:06:55] [INFO] 再探索パラメータの ES best_iter × 1.25 = 1001


INFO:42_lean113_retune_and_shrink:再探索パラメータの ES best_iter × 1.25 = 1001


config                 特徴量             列数 params      iters
--------------------------------------------------------------
T0_ref_lean113         lean113        113 A_PARAMS      560
T1_retuned_iter560     lean113        113 NEW           560
T1b_retuned_iterfit    lean113        113 NEW          1001
S1_no_agg33            no_agg33        33 A_PARAMS      560
S2_agg_mean49          agg_mean49      49 A_PARAMS      560


## D. 実行

In [84]:
# ============================================================
# 第D節: 実行
#   検証 : 先頭80%学習 / 生存者535名 / 8シード平均
#   提出 : Train全件学習 / 5シード平均（D3・40_と同一シード）
# ============================================================

def make_runner(label, spec):
    def _run():
        feats = cols_for(spec["features"], ag_train_80b)
        assert feats == cols_for(spec["features"], ag_full), "80%と全件で列が食い違っている"
        logger.info("=" * 60)
        logger.info(f"[{label}] {spec['features']} {len(feats)}列 / "
                    f"{spec['params_name']} / iterations={spec['iters']}")

        bi = best_iter_es(ag_train_80b, ag_val_surv, feats, spec["params"], TUNE_SEEDS)
        logger.info(f"  [診断] ES最適反復 ≈ {bi:.0f}")

        hold = fit_fixed(ag_train_80b, ag_val_surv, feats, spec["params"], spec["iters"], SEEDS_VAL)
        logger.info(f"  検証(生存者{len(ag_val_surv)}名): シード平均 {hold['val_seedavg']:.6f} "
                    f"/ 単一 {hold['val_single_mean']:.6f} ± {hold['val_single_sd']:.6f}")
        np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{label}_valpreds.npy", hold["val_preds"])

        tp = fit_full(ag_full, test_features_full, feats, spec["params"], spec["iters"], SEEDS_SUB)
        preds = tp.mean(axis=0)
        path = save_submission(test_features_full.index, preds, label)

        return make_row(config=label, feature_set=spec["features"], n_features=len(feats),
                        params_name=spec["params_name"], params=json.dumps(spec["params"]),
                        val_seedavg=hold["val_seedavg"], val_single_mean=hold["val_single_mean"],
                        val_single_sd=hold["val_single_sd"], best_iter_es=bi,
                        n_iterations=spec["iters"], n_train=len(ag_full),
                        pred_mean=float(preds.mean()), submission_path=path)
    return _run


results = {}
for _n, _s in CONFIGS.items():
    results[_n] = run_or_resume(_n, make_runner(_n, _s))

print()
print(f"{'config':<22s} {'列数':>5s} {'iters':>6s} {'val(8シード)':>14s} {'単一sd':>9s} {'予測平均':>9s}")
print("-" * 72)
for _n, _r in results.items():
    print(f"{_n:<22s} {int(_r['n_features']):>5d} {int(_r['n_iterations']):>6d} "
          f"{float(_r['val_seedavg']):>14.6f} {float(_r['val_single_sd']):>9.6f} "
          f"{float(_r['pred_mean']):>9.4f}")


[2026-08-12 01:06:55] [INFO] ============================================================


INFO:42_lean113_retune_and_shrink:============================================================


[2026-08-12 01:06:55] [INFO] [T0_ref_lean113] lean113 113列 / A_PARAMS / iterations=560


INFO:42_lean113_retune_and_shrink:[T0_ref_lean113] lean113 113列 / A_PARAMS / iterations=560


[2026-08-12 01:07:03] [INFO]   [診断] ES最適反復 ≈ 387


...

[2026-08-12 01:12:03] [INFO]     seed=99: 全件学習完了


INFO:42_lean113_retune_and_shrink:    seed=99: 全件学習完了


[2026-08-12 01:12:03] [INFO]   提出ファイル: 20260812_42_lean113_retune_and_shrink_S2_agg_mean49.csv（予測平均=0.5824）


INFO:42_lean113_retune_and_shrink:  提出ファイル: 20260812_42_lean113_retune_and_shrink_S2_agg_mean49.csv（予測平均=0.5824）



config                    列数  iters      val(8シード)      単一sd      予測平均
------------------------------------------------------------------------
T0_ref_lean113           113    560       0.514642  0.005220    0.5874
T1_retuned_iter560       113    560       0.517673  0.004527    0.5876
T1b_retuned_iterfit      113   1001       0.511253  0.005548    0.5892
S1_no_agg33               33    560       0.512949  0.007565    0.5758
S2_agg_mean49             49    560       0.508574  0.007139    0.5824


In [85]:
# ============================================================
# T0 の再現性チェック
#   T0 は 40_ R6_lean（Public 0.521729）と同じ特徴量・パラメータ・反復数・シード。
#   予測が一致しなければ以降の比較は無効。
# ============================================================

_t0 = pd.read_csv(results["T0_ref_lean113"]["submission_path"], header=None, names=[ID_COL, "pred"])
_r6 = sorted((PROJECT_ROOT / "data" / "output").glob("*/*_40_feature_reduction_R6_lean.csv"))

if _r6:
    _r6df = pd.read_csv(_r6[-1], header=None, names=[ID_COL, "pred"])
    _m = _t0.merge(_r6df, on=ID_COL, suffixes=("_t0", "_r6"))
    assert len(_m) == len(_t0), "社員IDが一致しない"
    _corr = _m["pred_t0"].corr(_m["pred_r6"])
    _mad = (_m["pred_t0"] - _m["pred_r6"]).abs().mean()
    print(f"R6_leanファイル: {_r6[-1].name}")
    print(f"  相関           : {_corr:.6f}")
    print(f"  平均絶対差     : {_mad:.6f}")
    print(f"  予測平均 T0/R6 : {_m['pred_t0'].mean():.4f} / {_m['pred_r6'].mean():.4f}")
    if _corr > 0.9999 and _mad < 0.001:
        print("✅ R6_lean(Public 0.521729)を再現できている")
    else:
        print("⚠️ 再現できていない。パイプラインの差分を特定すること")
else:
    print("⚠️ R6_leanの提出ファイルが見つからなかった（再現チェックをスキップ）")


R6_leanファイル: 20260811_40_feature_reduction_R6_lean.csv
  相関           : 1.000000
  平均絶対差     : 0.000000
  予測平均 T0/R6 : 0.5874 / 0.5874
✅ R6_lean(Public 0.521729)を再現できている


## E. 結果まとめ

In [86]:
# ============================================================
# 第E節: 結果まとめと提出方針
# ============================================================

_ref_val = float(results["T0_ref_lean113"]["val_seedavg"])
_ref_pred = pd.read_csv(results["T0_ref_lean113"]["submission_path"],
                        header=None, names=[ID_COL, "pred"]).set_index(ID_COL)["pred"]

rows = []
for name, r in results.items():
    p = pd.read_csv(r["submission_path"], header=None,
                    names=[ID_COL, "pred"]).set_index(ID_COL)["pred"].loc[_ref_pred.index]
    val = float(r["val_seedavg"])
    rows.append({
        "config": name, "特徴量": r["feature_set"], "列数": int(r["n_features"]),
        "params": r["params_name"], "iters": int(r["n_iterations"]),
        "val(生存者)": val, "T0との差": val - _ref_val,
        "T0との相関": float(np.corrcoef(p.values, _ref_pred.values)[0, 1]),
        "平均絶対差": float(np.abs(p.values - _ref_pred.values).mean()),
        "予測平均": float(p.mean()),
        "ファイル": Path(r["submission_path"]).name,
    })
summary = pd.DataFrame(rows)


def _verdict(row):
    if row["config"] == "T0_ref_lean113":
        return "不要（提出済み・参照用）"
    if row["T0との差"] > VAL_REJECT_MARGIN:
        return "見送り（足切り）"
    if row["params"] == "NEW" and not SUBMIT_TUNED:
        return "見送り（再探索の事前条件を満たさず）"
    return "提出する"


summary["提出"] = summary.apply(_verdict, axis=1)
pd.set_option("display.width", 240)
print(summary.to_string(index=False))
summary.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_summary.csv", index=False)
logger.info(f"サマリを保存: {TODAY}_{SCRIPT_NAME}_summary.csv")

print()
print("=" * 72)
print("提出順（現最良 R6_lean=T0 からの変更が小さい順）")
print("=" * 72)
_order = ["T2_lean113_itercurve", "T1_retuned_iter560", "T1b_retuned_iterfit",
          "S2_agg_mean49", "S1_no_agg33"]
_i = 0
for _c in _order:
    _row = summary[(summary["config"] == _c) & (summary["提出"] == "提出する")]
    if len(_row) == 0:
        continue
    _i += 1
    _r = _row.iloc[0]
    print(f"{_i}. {_c:<22s} {_r['ファイル']}")
    print(f"     {_r['列数']}列 / {_r['params']} / iters={_r['iters']} "
          f"/ T0との相関 {_r['T0との相関']:.5f} / 平均絶対差 {_r['平均絶対差']:.5f}")
if _i == 0:
    print("  提出対象なし。事前登録した条件をどれも満たさなかった。")

print()
print("※ T0との相関が0.9999超・平均絶対差0.002未満の構成は、38_のH1/H2と同じく")
print("   提出しても 0.521729 の近傍に戻るだけで情報が得られない可能性が高い。")


             config        特徴量  列数   params  iters  val(生存者)     T0との差   T0との相関    平均絶対差     予測平均                                                          ファイル                 提出
     T0_ref_lean113    lean113 113 A_PARAMS    560  0.514642  0.000000 1.000000 0.000000 0.587364      20260812_42_lean113_retune_and_shrink_T0_ref_lean113.csv       不要（提出済み・参照用）
 T1_retuned_iter560    lean113 113      NEW    560  0.517673  0.003030 0.982812 0.044417 0.587564  20260812_42_lean113_retune_and_shrink_T1_retuned_iter560.csv 見送り（再探索の事前条件を満たさず）
T1b_retuned_iterfit    lean113 113      NEW   1001  0.511253 -0.003390 0.995791 0.018784 0.589206 20260812_42_lean113_retune_and_shrink_T1b_retuned_iterfit.csv 見送り（再探索の事前条件を満たさず）
        S1_no_agg33   no_agg33  33 A_PARAMS    560  0.512949 -0.001693 0.954316 0.061233 0.575792         20260812_42_lean113_retune_and_shrink_S1_no_agg33.csv               提出する
      S2_agg_mean49 agg_mean49  49 A_PARAMS    560  0.508574 -0.006068 0.977236 0.042708 0.582442       2

INFO:42_lean113_retune_and_shrink:サマリを保存: 20260812_42_lean113_retune_and_shrink_summary.csv



提出順（現最良 R6_lean=T0 からの変更が小さい順）
1. S2_agg_mean49          20260812_42_lean113_retune_and_shrink_S2_agg_mean49.csv
     49列 / A_PARAMS / iters=560 / T0との相関 0.97724 / 平均絶対差 0.04271
2. S1_no_agg33            20260812_42_lean113_retune_and_shrink_S1_no_agg33.csv
     33列 / A_PARAMS / iters=560 / T0との相関 0.95432 / 平均絶対差 0.06123

※ T0との相関が0.9999超・平均絶対差0.002未満の構成は、38_のH1/H2と同じく
   提出しても 0.521729 の近傍に戻るだけで情報が得られない可能性が高い。


## F. 提出方針と結果の解釈

### 提出するファイル

第E節で「提出する」となったものを、**現最良（`R6_lean` = `T0`）からの変更が小さい順**に提出する。

| config | 何を検証するか |
|---|---|
| `T2_lean113_itercurve` | 113列では560反復が最適でないか（条件付きで生成） |
| `T1_retuned_iter560` | 113列向けに再探索したパラメータは効くか（変更はパラメータのみ） |
| `T1b_retuned_iterfit` | 同上（反復数も再探索パラメータ由来にした手続き一定版） |
| `S2_agg_mean49` | `agg` を mean だけにしてよいか |
| `S1_no_agg33` | **`agg` 80列を全部落として33列にしてよいか** |

`T0_ref_lean113` は `40_` R6_lean と同一内容なので提出しない（第D節の再現チェック用）。

### 結果の解釈ルール（事前登録）

- **Public < 0.521729** → その方向は正しい。次はその構成の上に積む
- **Public ≒ 0.5217 ± 0.001** → 差が無い。ただし**列数が少ない方／設定が単純な方を基準構成に採用**する
  （同じスコアなら軽い方が良い。以降の実験が速くなる）
- **Public > 0.5217** → その変更は害。方向を打ち切る

特に `S1_no_agg33` が横ばい以上なら、**441列 → 33列（92.5%削減）でスコアが落ちない**ことになり、
「このデータで実際に効いているのは入社時点の属性と転居×勤務地ミスマッチだけ」という
本プロジェクトの結論（`40_` 第61.5節のLOO診断）を Public で裏づけることになる。

### やらないこと

- **検証スコアで提出構成を選び直さない。** 分解能±0.011に対し見込む効果は0.002〜0.005で、
  原理的に判定できない
- **再探索パラメータと減量を同時に適用した構成を、このノートブックでは作らない。**
  どちらが効いたか分離できなくなる。両方がPublicで有効と分かってから次のノートブックで組む
- **`T1` 系を無条件に提出しない。** 第B節の事前条件（パラメータが実質的に異なる かつ
  改善幅がシードsdを超える）を満たさない場合は提出しない —— `38_` で
  「探索に使った検証セット上で有利に出るのは当然」と結論している

### 関連

- 現最良: `data/output/20260811/20260811_40_feature_reduction_R6_lean.csv`（Public 0.521729）
- 減量の本体: `src/40_feature_reduction.ipynb`、結果は `submit_result_report.md` 第61節
- `入社年` の外挿問題: `src/41_hire_year_extrapolation.ipynb`（`BASE="lean113"` で並行実行中）。
  **42_ と 41_ は同じ113列ベースで独立に評価できる**（41_は列を1つ引くだけ、42_はパラメータと減量）
